
#Implementación de MLP sobre datos sintéticos



In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import time

# Generación de curvas sintéticas

In [ ]:

import matplotlib.pyplot as plt

# ==============================
# Generación de curvas sintéticas
# ==============================
#



# Parámetros generales
n_objects = 3000

# Ventana temporal
T_max = 120.0


#Bandas de luz a simular
bands = ["g", "r", "i", "z"]

#Clases del dataset, básicamente una clasificación de ondas seno, ya que en eso se basa el dataset
# senos de alta/baja frecuencia, armónicos o señales con silencios
classes = {0:"sine_lowfreq", 1:"sine_highfreq", 2:"sine_harmonic", 3:"damped_sine"}

# semilla de aleatorización
rng = np.random.default_rng(42)
rows = []

# Factores por banda para simular correlación con longitud de onda
band_freq_factor = {"g": 1.2, "r": 1.0, "i": 0.9, "z": 0.8}
band_amp_factor = {"g": 1.1, "r": 1.0, "i": 0.9, "z": 0.8}
#Aqui comenzamos a generar cada dato.
for obj_id in range(n_objects):
    class_label = rng.choice(list(classes.keys()))

    # parámetros básicos de la onda, frecuencia, amplitud y fase
    # generados aleatoreamente
    freq = rng.uniform(0.02, 0.3)
    amp = rng.uniform(10, 100)
    phase = rng.uniform(0, 2*np.pi)

    # muestreo irregular en el tiempo
    n_obs = rng.integers(40, 200)
    times = np.sort(rng.uniform(0, T_max, size=n_obs))
    #asignacion de bandas y correlacion con parámetros de la onda
    for b in bands:
        # subset de tiempos por banda
        mask = rng.random(times.size) < rng.uniform(0.6, 1.0)
        t_band = times[mask]
        if len(t_band) == 0:
            continue

        # correlación por banda
        # Esto lo hice pues las distintas bandas de luz
        # se correlacionan con la forma de onda,
        # las bandas u (ultravioleta) por ejemplo tiene una frecuencia más alta que
        # las bandas y (infrarrojo)
        band_freq = freq * band_freq_factor[b]
        band_amp = amp * band_amp_factor[b]
        band_phase = phase + rng.uniform(-0.2, 0.2)
        band_baseline = rng.uniform(-30, 30)

        # generar flujo según clase
        if class_label == 0:
            flux = band_amp * np.sin(2*np.pi*band_freq*t_band + band_phase) + band_baseline
        elif class_label == 1:
            flux = band_amp * np.sin(2*np.pi*(band_freq*4)*t_band + band_phase) + band_baseline
        elif class_label == 2:
            flux = (band_amp * np.sin(2*np.pi*band_freq*t_band + band_phase) +
                    0.5*band_amp * np.sin(2*np.pi*2*band_freq*t_band)) + band_baseline
        else:
            damping = rng.uniform(0.001, 0.01)
            flux = band_amp * np.exp(-damping * t_band) * np.sin(2*np.pi*band_freq*t_band + band_phase) + band_baseline

        # ruido y errores para agregar realismo
        flux_err = np.abs(rng.normal(5.0, 2.0, size=len(flux))) + 0.02*np.abs(flux)
        flux_obs = flux + rng.normal(0, flux_err)

        for t, f, ferr in zip(t_band, flux_obs, flux_err):
            rows.append([obj_id, b, t, f, ferr, class_label])
    if (obj_id+1) % (n_objects//10) == 0:
        print(f"Progreso: {obj_id+1}/{n_objects} objetos generados ({(obj_id+1)/n_objects*100:.0f}%)")

# convertir a DataFrame
df = pd.DataFrame(rows, columns=["obj_id", "band", "time", "flux", "flux_err", "class_label"])
print("Generación completa. Dataset listo con", len(df), "filas.")
print(df.head())


Progreso: 300/3000 objetos generados (10%)
Progreso: 600/3000 objetos generados (20%)
Progreso: 900/3000 objetos generados (30%)
Progreso: 1200/3000 objetos generados (40%)
Progreso: 1500/3000 objetos generados (50%)
Progreso: 1800/3000 objetos generados (60%)
Progreso: 2100/3000 objetos generados (70%)
Progreso: 2400/3000 objetos generados (80%)
Progreso: 2700/3000 objetos generados (90%)
Progreso: 3000/3000 objetos generados (100%)
Generación completa. Dataset listo con 1151913 filas.
   obj_id band      time       flux   flux_err  class_label
0       0    g  0.883472 -65.754864   6.068019            0
1       0    g  2.593450  70.177730   8.917640            0
2       0    g  2.736465  91.380128   8.271817            0
3       0    g  3.698140  96.042521  10.648841            0
4       0    g  4.489507  40.301825   7.309155            0


## Identificación de Features

In [ ]:
# Verificación REAL de correlación entre datos y clases
print("=== VERIFICACIÓN REAL DE CORRELACIÓN ===\n")

# 1. Extraer características reales de los datos
features_list = []

for obj_id in df['obj_id'].unique():
    obj_data = df[df['obj_id'] == obj_id]
    obj_features = {'obj_id': obj_id, 'class_label': obj_data['class_label'].iloc[0]}

    # Características por banda 'g' (usemos solo una para simplificar)
    band_data = obj_data[obj_data['band'] == 'g']
    if len(band_data) > 1:
        flux = band_data['flux'].values
        time = band_data['time'].values

        # Características REALES que podrían diferenciar las clases
        obj_features['mean_flux'] = np.mean(flux)
        obj_features['std_flux'] = np.std(flux)
        obj_features['range_flux'] = np.max(flux) - np.min(flux)

        # Característica temporal: variación entre puntos consecutivos
        if len(flux) > 2:
            flux_diff = np.diff(flux)
            obj_features['mean_flux_change'] = np.mean(np.abs(flux_diff))
            obj_features['max_flux_change'] = np.max(np.abs(flux_diff))
        else:
            obj_features['mean_flux_change'] = 0
            obj_features['max_flux_change'] = 0

        features_list.append(obj_features)

# Crear DataFrame de características
features_df = pd.DataFrame(features_list)

# 2. Ver si las características difieren entre clases
print("ESTADÍSTICAS POR CLASE (valores reales de los datos):")
for class_id in range(4):
    class_data = features_df[features_df['class_label'] == class_id]
    print(f"\nClase {class_id} ({classes[class_id]}):")
    print(f"  Flujo promedio: {class_data['mean_flux'].mean():.2f} ± {class_data['mean_flux'].std():.2f}")
    print(f"  Desviación flujo: {class_data['std_flux'].mean():.2f} ± {class_data['std_flux'].std():.2f}")
    print(f"  Rango flujo: {class_data['range_flux'].mean():.2f} ± {class_data['range_flux'].std():.2f}")
    print(f"  Cambio medio flujo: {class_data['mean_flux_change'].mean():.2f} ± {class_data['mean_flux_change'].std():.2f}")

# 3. Verificación cuantitativa: ANOVA entre clases
from scipy import stats

print("\nVERIFICACIÓN ESTADÍSTICA (ANOVA):")
print("¿Las medias de las características son diferentes entre clases?")
print("(p-value < 0.05 significa que SÍ hay diferencias significativas)")

for feature in ['mean_flux', 'std_flux', 'range_flux', 'mean_flux_change']:
    groups = [features_df[features_df['class_label'] == i][feature] for i in range(4)]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"{feature}: p-value = {p_value:.6f} {'✓ DIFERENCIAS SIGNIFICATIVAS' if p_value < 0.05 else '✗ NO HAY DIFERENCIAS'}")

=== VERIFICACIÓN REAL DE CORRELACIÓN ===

ESTADÍSTICAS POR CLASE (valores reales de los datos):

Clase 0 (sine_lowfreq):
  Flujo promedio: -0.18 ± 18.18
  Desviación flujo: 42.19 ± 19.44
  Rango flujo: 138.63 ± 55.86
  Cambio medio flujo: 35.72 ± 20.04

Clase 1 (sine_highfreq):
  Flujo promedio: -0.73 ± 18.34
  Desviación flujo: 43.90 ± 20.62
  Rango flujo: 142.53 ± 58.71
  Cambio medio flujo: 47.59 ± 23.02

Clase 2 (sine_harmonic):
  Flujo promedio: -0.12 ± 18.71
  Desviación flujo: 46.44 ± 22.96
  Rango flujo: 162.26 ± 72.83
  Cambio medio flujo: 39.71 ± 22.59

Clase 3 (damped_sine):
  Flujo promedio: 0.24 ± 18.03
  Desviación flujo: 32.56 ± 15.59
  Rango flujo: 121.47 ± 51.72
  Cambio medio flujo: 27.96 ± 15.71

VERIFICACIÓN ESTADÍSTICA (ANOVA):
¿Las medias de las características son diferentes entre clases?
(p-value < 0.05 significa que SÍ hay diferencias significativas)
mean_flux: p-value = 0.785268 ✗ NO HAY DIFERENCIAS
std_flux: p-value = 0.000000 ✓ DIFERENCIAS SIGNIFICATIVAS
ran

# Función de visualización

In [ ]:
def show_curves(curves):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    fig.set_facecolor('white')

    epochs = np.arange(len(curves["val_loss"])) + 1

    ax[0].plot(epochs, curves['val_loss'], label='validation')
    ax[0].plot(epochs, curves['train_loss'], label='training')
    ax[0].set_xlabel('Epoch')
    ax[0].set_ylabel('Loss')
    ax[0].set_title('Loss evolution during training')
    ax[0].legend()

    ax[1].plot(epochs, curves['val_acc'], label='validation')
    ax[1].plot(epochs, curves['train_acc'], label='training')
    ax[1].set_xlabel('Epoch')
    ax[1].set_ylabel('Accuracy')
    ax[1].set_title('Accuracy evolution during training')
    ax[1].legend()

    plt.savefig("resultados.png")

# Preparación de datos para MLP

In [ ]:
# ==============================
# 1. PREPARACIÓN DE DATOS CON 4 BANDAS
# ==============================

class LightCurveDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

def prepare_mlp_data_4bands(df):
    """Extrae características de las 4 BANDAS para MLP"""
    features_list = []

    for obj_id in df['obj_id'].unique():
        obj_data = df[df['obj_id'] == obj_id]
        obj_features = []

        # Para CADA banda extraer características (g, r, i, z)
        for band in ['g', 'r', 'i', 'z']:
            band_data = obj_data[obj_data['band'] == band]
            if len(band_data) > 1:
                flux = band_data['flux'].values

                # Las 3 características buenas por cada banda
                obj_features.extend([
                    np.std(flux),                    # std_flux
                    np.max(flux) - np.min(flux),     # range_flux
                    np.mean(np.abs(np.diff(flux))) if len(flux) > 1 else 0,  # mean_flux_change
                ])
            else:
                # Si no hay datos en esa banda, llenar con ceros
                obj_features.extend([0, 0, 0])

        features_list.append(obj_features)

    return np.array(features_list)

# Preparar los datos CON LAS 4 BANDAS
print("Preparando datos para MLP con 4 bandas...")
X = prepare_mlp_data_4bands(df)
y = df.groupby('obj_id')['class_label'].first().values

print(f"Shape de características: {X.shape}")
print(f"Shape de labels: {y.shape}")

# Dividir en train/val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalizar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Preparando datos para MLP con 4 bandas...
Shape de características: (3000, 12)
Shape de labels: (3000,)
Train: (2400, 12), Val: (600, 12)


# Definición del Modelo MLP

In [ ]:
# ==============================
# 2. DEFINICIÓN DEL MODELO MLP
# ==============================

class MLPModel(nn.Module):
    def __init__(self, input_size, num_classes, dropout_p=0.3):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        return self.net(x)

## Función de Entrenamiento

In [ ]:
# ==============================
# 3. FUNCIONES DE ENTRENAMIENTO
# ==============================

def train_step(x_batch, y_batch, model, optimizer, criterion, use_gpu):
    # Predicción
    y_predicted = model(x_batch)

    # Cálculo de loss
    loss = criterion(y_predicted, y_batch)

    # Actualización de parámetros
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return y_predicted, loss

def evaluate(val_loader, model, criterion, use_gpu):
    cumulative_loss = 0
    cumulative_predictions = 0
    data_count = 0

    model.eval()
    with torch.no_grad():
        for x_val, y_val in val_loader:
            if use_gpu:
                x_val = x_val.cuda()
                y_val = y_val.cuda()

            y_predicted = model(x_val)
            loss = criterion(y_predicted, y_val)

            class_prediction = torch.argmax(y_predicted, axis=1)
            cumulative_predictions += (y_val == class_prediction).sum().item()
            cumulative_loss += loss.item()
            data_count += y_val.shape[0]

    val_acc = cumulative_predictions / data_count
    val_loss = cumulative_loss / len(val_loader)

    return val_acc, val_loss

def train_model(model, train_dataset, val_dataset, epochs, criterion,
                batch_size=32, lr=0.001, n_evaluations_per_epoch=4,
                use_gpu=False):

    if use_gpu:
        model.cuda()

    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

    # Optimizador
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Listas para curvas
    curves = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": [],
        "val_loss": [],
    }

    t0 = time.perf_counter()

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        # Entrenamiento
        model.train()
        cumulative_train_loss = 0
        cumulative_train_corrects = 0
        train_count = 0

        for i, (x_batch, y_batch) in enumerate(train_loader):
            if use_gpu:
                x_batch = x_batch.cuda()
                y_batch = y_batch.cuda()

            y_predicted, loss = train_step(x_batch, y_batch, model, optimizer, criterion, use_gpu)

            cumulative_train_loss += loss.item()
            train_count += 1

            # Calcular accuracy
            class_prediction = torch.argmax(y_predicted, axis=1)
            cumulative_train_corrects += (y_batch == class_prediction).sum().item()

            # Evaluación intermedia
            if (i % (len(train_loader) // n_evaluations_per_epoch) == 0) and (i > 0):
                train_loss = cumulative_train_loss / train_count
                train_acc = cumulative_train_corrects / (batch_size * train_count)
                print(f"  Batch {i}/{len(train_loader)} - Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")

        # Evaluación final de la época
        val_acc, val_loss = evaluate(val_loader, model, criterion, use_gpu)

        train_loss = cumulative_train_loss / len(train_loader)
        train_acc = cumulative_train_corrects / len(train_dataset)

        print(f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")
        print(f"Val loss: {val_loss:.4f}, Val acc: {val_acc:.4f}")

        # Guardar curvas
        curves["train_acc"].append(train_acc)
        curves["val_acc"].append(val_acc)
        curves["train_loss"].append(train_loss)
        curves["val_loss"].append(val_loss)

    print(f"\nTiempo total de entrenamiento: {time.perf_counter() - t0:.2f} [s]")
    model.cpu()

    return curves


In [ ]:
# ==============================
# 4. ENTRENAMIENTO
# ==============================

# Crear datasets
train_dataset = LightCurveDataset(X_train, y_train)
val_dataset = LightCurveDataset(X_val, y_val)

# Hiperparámetros
input_size = X_train.shape[1]  # 12 características
num_classes = len(np.unique(y))  # 4 clases
epochs = 50
batch_size = 32
lr = 0.001

# Modelo y loss
model = MLPModel(input_size, num_classes, dropout_p=0.0)
criterion = nn.CrossEntropyLoss()

print(f"Input size: {input_size}, Num classes: {num_classes}")
print(f"Model architecture: {model}")

# Entrenar
curves = train_model(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    epochs=epochs,
    criterion=criterion,
    batch_size=batch_size,
    lr=lr,
    use_gpu=torch.cuda.is_available()
)
show_curves(curves)

Input size: 12, Num classes: 4
Model architecture: MLPModel(
  (net): Sequential(
    (0): Linear(in_features=12, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.0, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.0, inplace=False)
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): ReLU()
    (8): Linear(in_features=16, out_features=4, bias=True)
  )
)


AttributeError: 'numpy.ndarray' object has no attribute 'perf_counter'

# Preparacion de datos para CNN


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from scipy.interpolate import interp1d

class LightCurveCNNDataset(Dataset):
    def __init__(self, df, bands=["g","r","i","z"], seq_len=100):
        self.df = df
        self.obj_ids = df["obj_id"].unique()
        self.bands = bands
        self.seq_len = seq_len
        self.T_min, self.T_max = df["time"].min(), df["time"].max()

    def __len__(self):
        return len(self.obj_ids)

    def __getitem__(self, idx):
        obj_id = self.obj_ids[idx]
        obj_data = self.df[self.df["obj_id"] == obj_id]
        label = obj_data["class_label"].iloc[0]

        # grilla temporal uniforme
        t_grid = np.linspace(self.T_min, self.T_max, self.seq_len)

        flux_channels = []
        for b in self.bands:
            band_data = obj_data[obj_data["band"] == b]
            if len(band_data) > 1:
                f = interp1d(band_data["time"], band_data["flux"],
                             bounds_error=False, fill_value="extrapolate")
                flux_resampled = f(t_grid)
            else:
                flux_resampled = np.zeros(self.seq_len)

            # normalizar por banda
            flux_resampled = (flux_resampled - np.mean(flux_resampled)) / (np.std(flux_resampled)+1e-8)
            flux_channels.append(flux_resampled)

        x = np.stack(flux_channels)  # shape: (n_bands, seq_len)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(label, dtype=torch.long)


# Modelo CNN


In [ ]:
import torch.nn as nn

class CNN1DModel(nn.Module):
    def __init__(self, n_channels=4, num_classes=4, seq_len=100):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # reduce a (batch, 128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.classifier(x)


# Entrenamiento CNN

In [ ]:
# Crear dataset y dataloaders
dataset = LightCurveCNNDataset(df, seq_len=100)
n_train = int(0.8 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, len(dataset)-n_train])

# Crear modelo
model = CNN1DModel(n_channels=4, num_classes=4)

# Entrenar con tus mismas funciones
curves = train_model(
    model=model,
    train_dataset=train_ds,
    val_dataset=val_ds,
    epochs=30,
    criterion=nn.CrossEntropyLoss(),
    batch_size=32,
    lr=1e-3,
    use_gpu=torch.cuda.is_available()
)


In [ ]:
show_curves(curves)